# Module 00 — Build with AI

**Released:** April 2026 — [docs.crewai.com/en/guides/coding-tools/build-with-ai](https://docs.crewai.com/en/guides/coding-tools/build-with-ai)

"Build with AI" is CrewAI's bet that most new projects this year will be scaffolded by coding agents (Claude Code, Cursor, Codex, Gemini CLI, Windsurf). The framework now ships **installable coding-agent skills** and **machine-readable docs** so those agents can scaffold, build, and deploy CrewAI projects without guessing your conventions.

This module is the **entry point** for the rest of the tour. There is no Flow code here — it's about the meta-layer: installing the skills, then letting your coding agent take you from an empty directory to a working Flow.

## The two pieces

1. **Coding-agent skills** — installable skill packs from `crewAIInc/skills` that teach your coding agent CrewAI's patterns (Flows, Crews, agents, tasks, deploy). The `ask-docs` skill in the pack queries `https://docs.crewai.com/llms.txt` over MCP, so doc lookups happen automatically.
2. **`crewai deploy`** — the CLI knows how to push a local project to [CrewAI AMP](https://app.crewai.com) for hosted traces, auth, A2A hosting, and scaling.

## 1. Install the Build with AI skills

These are **coding-agent skills** — they live in your coding agent's skill directory and make the agent itself CrewAI-literate. (They are separate from this repo's runtime `skills/` — see the disambiguation cell below.)

### Claude Code (plugin marketplace)

```bash
/plugin marketplace add crewAIInc/skills
/plugin install crewai-skills@crewai-plugins
/reload-plugins
```

### Any other agent — Cursor, Codex, Gemini CLI, Windsurf (universal installer)

```bash
npx skills add crewaiinc/skills
```

The `npx` path writes the same skill packs into the skill directory your IDE watches (Cursor → `.cursor/skills/`, Codex → `~/.codex/skills/`, etc.). After it finishes, restart your agent session so the skills load.

### What you get

Four skills auto-activate based on what you ask:

| Skill | Auto-fires when you… |
|---|---|
| `getting-started` | Scaffold a new project, pick Flow vs Crew, wire `crew.py`/`main.py`. |
| `design-agent` | Author `Agent` objects (role / goal / backstory / tools / LLM). |
| `design-task` | Write `Task` objects with tight `description` / `expected_output`. |
| `ask-docs` | Ask framework questions — the agent consults `llms.txt` instead of guessing. |

The `ask-docs` skill queries `https://docs.crewai.com/llms.txt` via MCP, so the agent always consults current docs — no extra wiring needed.

## Two kinds of skills — don't mix them up

CrewAI uses the word "skill" in two distinct places. Both are `SKILL.md` files with frontmatter, but they target different consumers:

| | **Coding-agent skills** (this module) | **Runtime agent skills** (Module 01) |
|---|---|---|
| Who reads them | Your coding agent (Claude Code, Cursor, …) | A CrewAI `Agent` at runtime |
| Where they live | `~/.claude/skills/`, `.cursor/skills/`, … | `skills/<name>/` in your project |
| How installed | `npx skills add crewaiinc/skills` or Claude plugin | Referenced via `Agent(skills=[path])` |
| Purpose | Teach the *scaffolding agent* CrewAI conventions | Inject instructions into the *running agent* |

This repo's `skills/researcher-skill` and `skills/citation-skill` are the **runtime** kind — Module 01 loads them with `Agent(skills=[...])`. They are not meant to be copied into your coding agent.

In [ ]:
from pathlib import Path

# Preview the runtime SKILL.md packs this repo ships for Module 01.
# These are loaded by `Agent(skills=[...])` in src/showcase/flows/skills_flow.py.
skills_dir = Path.cwd().parent / "skills"
for s in sorted(p for p in skills_dir.iterdir() if p.is_dir()):
    print(f"- {s.name}")

## 2. Walk-through — scaffold a Deep Research Flow

This is the payoff. With the skills installed, you open your coding agent in an empty directory and **talk to it in English**. The skills pull the right CrewAI knowledge in as you go.

We'll build a **Deep Research Flow** — a 2-agent Crew wrapped in a Flow that takes a topic, searches the web via Exa's remote MCP server, and produces a cited brief. Every sample in this repo wraps a Flow (`crewai create flow …`), so this walkthrough matches the shape of the five modules that follow.

### Step 1 — Prereqs & open your coding agent

```bash
mkdir deep-research && cd deep-research

# Grab an Exa key (free tier works): https://dashboard.exa.ai/api-keys
export EXA_API_KEY=sk-exa-...
export ANTHROPIC_API_KEY=sk-ant-...

claude           # or: cursor . / codex / gemini
```

Confirm the skills loaded. In Claude Code: `/plugin` should list `crewai-skills`. In Cursor: the Skills panel should show the four entries from the table above.

### Step 2 — Ask the agent to scaffold a Flow

Type something like:

> *Scaffold a new CrewAI Flow project called `deep_research`. Two agents inside: a `Researcher` that uses the Exa MCP server to search the web, and a `Synthesizer` that turns the findings into a cited markdown brief. The Flow takes a `topic` input and returns the brief.*

The `getting-started` skill fires and the agent runs:

```bash
crewai create flow deep_research
cd deep_research
```

`crewai create flow` generates the canonical layout — a top-level Flow with a nested Crew:

```
deep_research/
├── src/deep_research/
│   ├── crews/
│   │   └── research_crew/
│   │       ├── config/
│   │       │   ├── agents.yaml        # Researcher + Synthesizer
│   │       │   └── tasks.yaml         # research → synthesize
│   │       └── research_crew.py       # MCPServerAdapter wiring goes here
│   ├── main.py                        # DeepResearchFlow(Flow[State])
│   └── tools/
├── pyproject.toml
└── .env
```

### Step 3 — Wire up the Exa MCP server

Now ask:

> *In `research_crew.py`, connect the Researcher to the Exa remote MCP server over streamable-http. Use the `exaApiKey` query-param auth scheme.*

The `ask-docs` skill pulls the MCP-integration page from `llms.txt`, and the agent writes something like:

```python
# src/deep_research/crews/research_crew/research_crew.py
import os
from crewai import Agent, Crew, Process, Task
from crewai.project import CrewBase, agent, crew, task
from crewai_tools import MCPServerAdapter


EXA_SERVER = {
    "url": f"https://mcp.exa.ai/mcp?exaApiKey={os.environ['EXA_API_KEY']}",
    "transport": "streamable-http",
}


@CrewBase
class ResearchCrew:
    agents_config = "config/agents.yaml"
    tasks_config = "config/tasks.yaml"

    @agent
    def researcher(self) -> Agent:
        # MCPServerAdapter returns the tool list; passing the adapter as a
        # context manager would close it early, so we open it at kickoff.
        with MCPServerAdapter(EXA_SERVER) as tools:
            self._exa_tools = tools  # held for the crew lifetime
        return Agent(config=self.agents_config["researcher"], tools=self._exa_tools)

    @agent
    def synthesizer(self) -> Agent:
        return Agent(config=self.agents_config["synthesizer"])

    @task
    def research(self) -> Task:
        return Task(config=self.tasks_config["research"])

    @task
    def synthesize(self) -> Task:
        return Task(config=self.tasks_config["synthesize"])

    @crew
    def crew(self) -> Crew:
        return Crew(agents=self.agents, tasks=self.tasks, process=Process.sequential)
```

Exa's MCP server exposes `web_search_exa` (clean, LLM-ready page content) and `web_fetch_exa` (full page by URL) by default. The Researcher gets both as CrewAI tools automatically.

### Step 4 — Define the agents, tasks, and Flow state

> *Fill in `agents.yaml`, `tasks.yaml`, and `main.py`. The Researcher should decompose the topic into sub-questions before searching; the Synthesizer should emit `## Findings` and `## References` sections. Put the `topic` on Flow state.*

`design-agent` + `design-task` shape the YAML:

```yaml
# config/agents.yaml
researcher:
  role: Deep Research Analyst
  goal: Gather primary sources on {topic} and extract evidence-backed findings
  backstory: >
    Former investigative journalist. Decomposes any topic into atomic sub-questions
    and hunts down primary sources before summarizing.
  llm: anthropic/claude-sonnet-4-6

synthesizer:
  role: Research Synthesizer
  goal: Compile findings into a tight, well-cited brief
  backstory: Writes the one-page memo that a busy exec actually reads.
  llm: anthropic/claude-sonnet-4-6
```

```yaml
# config/tasks.yaml
research:
  description: >
    Decompose {topic} into 3–5 sub-questions. For each, use web_search_exa to find
    primary sources and web_fetch_exa to pull key passages. Capture URLs verbatim.
  expected_output: A list of findings, each with a supporting URL.
  agent: researcher

synthesize:
  description: Turn the findings into a 300-word cited brief on {topic}.
  expected_output: Markdown with '## Findings' and '## References' sections.
  agent: synthesizer
  context: [research]
```

`getting-started` wires the Flow:

```python
# src/deep_research/main.py
from crewai.flow import Flow, listen, start
from pydantic import BaseModel

from deep_research.crews.research_crew.research_crew import ResearchCrew


class ResearchState(BaseModel):
    topic: str = ""
    brief: str = ""


class DeepResearchFlow(Flow[ResearchState]):
    @start()
    def require_topic(self) -> None:
        if not self.state.topic:
            raise ValueError("Pass a topic via `crewai run -- topic=...`")

    @listen(require_topic)
    def run_research(self) -> str:
        result = ResearchCrew().crew().kickoff(inputs={"topic": self.state.topic})
        self.state.brief = str(result)
        return self.state.brief


def kickoff() -> str:
    flow = DeepResearchFlow()
    flow.kickoff()
    return flow.state.brief


if __name__ == "__main__":
    print(kickoff())
```

### Step 5 — Install deps and run

```bash
crewai install                                   # resolves pyproject.toml (incl. crewai_tools for MCP)
crewai run -- topic="the 2026 state of open-source A2A protocol adoption"
```

`crewai run` invokes `main.py` → `DeepResearchFlow.kickoff()`. You'll see the Researcher stream Exa search hits, then the Synthesizer produce a markdown brief with a `## References` section listing the URLs Exa returned.

### Step 6 — Iterate through the agent

Once it runs, keep going through the coding agent — don't hand-edit YAML again. Some natural next asks:

- *"Add a `Critic` agent that flags any claim without a citation."* → `design-agent` + `design-task` fire together.
- *"Turn on Exa's `web_search_advanced_exa` tool so the Researcher can filter by domain/date."* → `ask-docs` recalls that Exa enables extra tools via the `?tools=` query param.
- *"Checkpoint the Flow after the research step so we can re-run synthesis without re-searching."* → `getting-started` rewrites `main.py` using the pattern from [Module 04 — Checkpointing](./04_checkpointing.ipynb).
- *"Expose this Flow as an A2A server."* → `ask-docs` pulls the A2A config straight from `llms.txt` (covered in [Module 05](./05_a2a.ipynb)).

This is the loop the Build with AI skills are optimised for: you stay in plain English, the agent stays in current CrewAI idioms.

## 3. Deploy to AMP

Once the Flow runs locally, ship it:

```bash
crewai deploy --prepare   # generate deployment config
crewai deploy             # push to CrewAI AMP
```

AMP gives you: hosted traces, per-run observability, enterprise auth (SSO / Keycloak), A2A hosting, and horizontal scaling. Deployed Flows expose a REST API so other services can call them.

## 4. A `CLAUDE.md` you can crib

Drop something like this at the repo root so every future coding-agent session starts with the right context:

```markdown
# CrewAI Project

## Docs (read first)
- Index: https://docs.crewai.com/llms.txt

## Skills
- Coding-agent skills are installed via `npx skills add crewaiinc/skills`
  (provides getting-started, design-agent, design-task, ask-docs).
- Runtime SKILL.md packs live in `skills/` and are loaded by `Agent(skills=[...])`.

## Conventions
- Every runtime use case wraps a Flow — always scaffold with `crewai create flow <name>`.
- Default LLM: `anthropic/claude-sonnet-4-6` (see `src/showcase/shared/llm.py`).
- Remote tools come in over MCP streamable-http via `MCPServerAdapter` from `crewai_tools`.
- Long-running work is checkpointed (see `notebooks/04_checkpointing.ipynb`).

## Deploy
- `crewai deploy --prepare` → `crewai deploy`. Do not hand-roll deployment.
```

## Next steps

- **[Module 01 → Agent Skills](./01_agent_skills.ipynb)** — the runtime `SKILL.md` primitive.
- **[Module 02 → Plan-and-Execute](./02_plan_and_execute.ipynb)** — turn fuzzy asks into ordered work.
- **[Module 03 → Unified Memory](./03_unified_memory.ipynb)** — persistent context across steps.
- **[Module 04 → Checkpointing](./04_checkpointing.ipynb)** — resume long runs.
- **[Module 05 → A2A](./05_a2a.ipynb)** — agents calling agents.